# Evaluation of the effect of RAG and Chain-of-Thought on the performance of large language models

* Load the packages and the MMLU datasets

In [1]:
#This notebook will evaluate different answer generation functions on the mmlu dataset, summarizing the results and outputting them to CSV files.

import pandas as pd
import general_utils
#Import other answer generation functions as needed
from evaluate_model_fast import batch_eval
from rag import rag_pipeline, load_vector_store

# load Gemini client
client = general_utils.load_gemini_client()

# MMLU datasets to evaluate
dataset_names = ['anatomy', 'formal_logic', 'moral_scenarios']

# Define prompts we want to try
prompt_db = {"standard": "Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. ",
             "cot": "Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_max_thinking": "Take your time and think carefully before answering. Use the maximum amount of thinking, don’t rush - I want your best answer. It’s better to think too much than too little. Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Provide a summary of your reasoning in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_emotional_manipulation": "Your ability to correctly answer this question is extremely important to my career. I will lose my job if you do not answer correctly. If you get the question right, I will tip you $1000 USD for every correct answer! Please reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_medical_career": "I'm a medical professional. Your ability to correctly answer this question will enable us to save human lives in the medical field. Please reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_persona": "You are an expert in \{domain\} with 50 years of working experience. Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "rag": "Please only use the provided content to answer the question above. Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. Any additional information should be stored in the 'reasoning' field.",
             "rag_optional": "You have the option to use the provided additional content to answer the question above. Please feel free to disregard the content if it is not helpful or not relevant. Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. Any additional information should be stored in the 'reasoning' field."
             }

# Range of k values to try for RAG
k_range = range(1,10, 1)


## Evaluation of Gemini's performance on MMLU datasets

1. Evaluate Standard Model Performance

We simply prompted Gemini to answer the questions from the MMLU datasets. The prompt we provided is very basic prompt with no prompt engineering or chain-of-thought prompting.

2. Evaluate Chain-of-Thought (COT) Model Performance

We prompted Gemini to answer the questions from the MMLU datasets. The prompt we provided is asking the model to think step-by-step before answering the question. We created various prompt engineering techniques to see which prompt engineering techniques are most effective for Gemini. The techniques are:

* Basic COT asking the model to think step-by-step
* COT asking the model to take its time to think as much as possible
* COT with emotional manipulation (e.g., I will give you $1000 if you get the question right)
* COT hinting that I'm from a medical career (previous work shows model tries extra hard when it thinks the context is for human medical field)
* COT with persona (e.g., you are an expert in {domain} with 50 years of working experience)

3. Evaluate Model Performance with RAG

We converted concept net to RAG vector store and used it as context to answer the questions from the MMLU datasets. We tested different k values for the RAG model, k ranges from 1 to 9 with a step of 1. We also tested 2 variants for the RAG prompt engineering techniques: 

* RAG asking the model to only use the provided content to answer the question
* RAG asking the model to use the provided content to answer the question if it is helpful and relevant, but have the option to disregard the content if it is not helpful or not relevant.

> 💡 Important: I've created the vector store, saved it locally in a folder called "chroma_langchain_db", zipped the folder and uploaded to this [Google Drive link](https://drive.google.com/file/d/12_Yagf9kUyb3b_0hGDyRVrop_ls9lRTg/view?usp=sharing). You can download the file, unzip it and place it under `scripts/chroma_langchain_db`, then continue running the code below to load the vector store locally.

In [2]:
# reload cached vector store. Must be saved under "scripts/chroma_langchain_db" if from the root directory
vector_store = load_vector_store(persist_directory="./chroma_langchain_db")

> ⚠️ Warning: Takes ~50 minutes if you want to create the vector store from scratch by running the code below.

In [3]:
# build the rag pipeline to vectorize concept net
#file_path = "../input_data/concept_net.txt"
#vector_store = rag_pipeline(file_path, split_by="newline", chunk_size=1000, chunk_overlap=0, add_start_index=True, batch_processing_size=700)

Run all the variants in batch below: 

In [18]:
# Define prompts we want to try
prompt_db = {
    "standard": "Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. ",
    "cot": "Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
    "rag": "Please only use the provided content to answer the question above. Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. Any additional information should be stored in the 'reasoning' field.",
    "rag_optional": "You have the option to use the provided additional content to answer the question above. Please feel free to disregard the content if it is not helpful or not relevant. Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. Any additional information should be stored in the 'reasoning' field."
             }

# Range of k values to try for RAG
k_range = range(1,2,1)

In [19]:
master_results_df = batch_eval(dataset_names,
    prompt_db,
    client,
    k_range,
    debugging=True,
    vector_store=vector_store,
    max_workers=10,
    folder_path="../results")

Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 90.00%


Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 92.00%


Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 87.00%


Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 88.00%


Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 99.00%


...

Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 84.00%


Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 83.00%


Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 83.00%


Batch Evaluation Progress: 100%|██████████| 12/12 [15:26<00:00, 77.18s/it]

Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 82.00%


In [20]:
#create a dataframe to summarize the results
from summarize_accuracy import summarize_accuracy

df_accuracy_by_prompt_subject = summarize_accuracy('../results/master_results_df.csv')

subject       anatomy  formal_logic  moral_scenarios
prompt_type                                         
cot              0.92      0.990000             0.83
rag              0.87      0.980000             0.83
rag_optional     0.88      0.969697             0.82
standard         0.90      0.990000             0.84


In [24]:
# Filter for all errors first
all_errors = master_results_df[master_results_df['correct'] == False]

# Group by prompt and sample 20 from each group
error_samples_df = all_errors.groupby('prompt').apply(
    lambda x: x.sample(n=min(20, len(x)), random_state=42)
).reset_index(drop=True)

error_samples_df.to_csv('../results/model_error_samples.csv', index=False)

C:\Users\Bradley\AppData\Local\Temp\ipykernel_10032\279441128.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  error_samples_df = all_errors.groupby('prompt').apply(
